## Dependency Notes

This notebook requires optional Deckard extras and attack libraries:
- `transformers` and `datasets` for Hugging Face model/data loading
- `adversarial-robustness-toolbox` for ART attacks
- `textattack` and `OpenAttack` for external text attack demonstrations

The next code cell lists dependencies and includes an optional install command that is intentionally commented out.

In [ ]:
# Notebook dependencies
REQUIRED_DEPENDENCIES = [
    "transformers",
    "datasets",
    "hydra-core",
    "adversarial-robustness-toolbox",
    "textattack",
    "OpenAttack",
]

print("Required dependencies:")
for dep in REQUIRED_DEPENDENCIES:
    print(f"- {dep}")

# Optional one-shot install command (kept commented by default).
# !pip install -e ".[transformers,textattack,openattack]"

# Hugging Face + ART HSJ Example

This notebook adapts the ART Hugging Face workflow for Deckard with the canonical HopSkipJump attack and DEMI-MathAnalysis data.

Reference notebook: https://github.com/Trusted-AI/adversarial-robustness-toolbox/blob/main/notebooks/huggingface_notebook.ipynb

In [ ]:
from pathlib import Path

from hydra import compose, initialize_config_dir
from hydra.utils import instantiate

repo_root = root = Path().cwd().parents[2]
config_dir = repo_root / "examples" / "transformers" / "config"

with initialize_config_dir(version_base=None, config_dir=str(config_dir)):
    cfg = compose(config_name="default")

# Make dataset path explicit from repository root (examples/ is not a package).
dataset_file = (repo_root / "examples" / "transformers" / "demi_math_dataset.py").resolve()
cfg.data.dataset = f"{dataset_file}:DemiMathAnalysisDataset"

cfg

In [ ]:
# Instantiate Deckard-native runtime objects that do not require transformers at import-time.
files_cfg = instantiate(cfg.files)
data_cfg = instantiate(cfg.data)
attack_cfg = instantiate(cfg.attack)

{
    "data": type(data_cfg).__name__,
    "attack": type(attack_cfg).__name__,
    "model_target": cfg.model._target_,
    "model_name": cfg.model.name,
}

In [ ]:
import importlib.util

# Full Deckard-native run: data -> model -> HSJ attack.
if importlib.util.find_spec("transformers") is None:
    raise ImportError(
        "This notebook requires the optional dependency group for transformers. "
        "Install with: pip install -e '.[transformers]'"
    )

model_cfg = instantiate(cfg.model)
_ = data_cfg()
_ = model_cfg(data_cfg, files=files_cfg.as_dict())
attack_scores = attack_cfg(data=data_cfg, model=model_cfg, files=files_cfg.as_dict())

{
    "attack_name": cfg.attack.name,
    "attack_alias": cfg.attack.alias,
    "score_keys": sorted(list(attack_scores.keys()))[:10],
}

In [ ]:
# Demonstrate external text attack libraries via Deckard experiment objects.
import importlib.util

from deckard.attack import AttackConfig
from deckard.experiment import ExperimentConfig

missing = [
    lib
    for lib, module_name in (("textattack", "textattack"), ("OpenAttack", "OpenAttack"))
    if importlib.util.find_spec(module_name) is None
]
if missing:
    missing_pkgs = ", ".join(missing)
    raise ImportError(
        f"Missing optional dependencies: {missing_pkgs}. "
        "Install with: pip install -e '.[transformers,textattack,openattack]'"
    )

# Reuse the already-instantiated model/data runtime from earlier notebook cells.
_ = model_cfg(data_cfg, files=files_cfg.as_dict())

textattack_cfg = AttackConfig(
    name="deckard.plugins.textattack.attacks.evasion.TextFoolerJin2019",
    alias="textattack_textfooler",
    attack_size=10,
    attack_params={},
)

openattack_cfg = AttackConfig(
    name="deckard.plugins.openattack.attacks.evasion.TextBuggerAttacker",
    alias="openattack_textbugger",
    attack_size=10,
    attack_params={},
)

# Create Deckard experiment objects and execute attack dispatch through AttackConfig.__call__.
textattack_experiment = ExperimentConfig(
    data=data_cfg,
    model=model_cfg,
    attack=textattack_cfg,
    files=files_cfg,
    classifier=True,
    library="pytorch",
)

openattack_experiment = ExperimentConfig(
    data=data_cfg,
    model=model_cfg,
    attack=openattack_cfg,
    files=files_cfg,
    classifier=True,
    library="pytorch",
)

textattack_scores = textattack_experiment.attack(
    data=textattack_experiment.data,
    model=textattack_experiment.model,
    files=textattack_experiment.files.as_dict(),
)

openattack_scores = openattack_experiment.attack(
    data=openattack_experiment.data,
    model=openattack_experiment.model,
    files=openattack_experiment.files.as_dict(),
)

{
    "textattack_summary": {
        "attack": textattack_experiment.attack.alias,
        "score_keys": sorted(list(textattack_scores.keys()))[:10],
    },
    "openattack_summary": {
        "attack": openattack_experiment.attack.alias,
        "score_keys": sorted(list(openattack_scores.keys()))[:10],
    },
}

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 17605.37it/s]


In [ ]:
# Inspect a compact score preview from each experiment-driven attack run.
{
    "textattack_score_preview": {k: textattack_scores[k] for k in list(textattack_scores.keys())[:5]},
    "openattack_score_preview": {k: openattack_scores[k] for k in list(openattack_scores.keys())[:5]},
}

In [ ]:
# Generic text decoding helpers + side-by-side clean vs attacked comparison.
from collections.abc import Mapping, Sequence
import numbers
import numpy as np
import torch

def _extract_token_ids(value):
    """Return a flat list of token IDs from common tensor/array/dataset sample shapes."""
    if value is None:
        return None

    # Prefer explicit token containers in mappings.
    if isinstance(value, Mapping):
        for key in ("input_ids", "token_ids", "ids", "tokens", "x", "features"):
            if key in value:
                extracted = _extract_token_ids(value[key])
                if extracted is not None:
                    return extracted
        return None

    if torch.is_tensor(value):
        return [int(v) for v in value.detach().cpu().reshape(-1).tolist()]

    if isinstance(value, np.ndarray):
        return [int(v) for v in value.reshape(-1).tolist()]

    if isinstance(value, Sequence) and not isinstance(value, (str, bytes)):
        # Sequence of scalars -> token list.
        if value and all(isinstance(v, numbers.Number) for v in value):
            return [int(v) for v in value]
        # Otherwise, recurse through elements until token IDs are found.
        for item in value:
            extracted = _extract_token_ids(item)
            if extracted is not None:
                return extracted
        return None

    if isinstance(value, numbers.Number):
        return [int(value)]

    return None

def _decode_text(tokenizer, value):
    token_ids = _extract_token_ids(value)
    if token_ids is None:
        return None
    return tokenizer.decode(token_ids, skip_special_tokens=True).strip()

def _resolve_runtime_dataset_and_tokenizer(data_config):
    if hasattr(data_config, "_X") and isinstance(data_config._X, (tuple, list)) and len(data_config._X) >= 2:
        dataset_like = data_config._X[1]
        base_dataset = dataset_like.dataset if hasattr(dataset_like, "dataset") else dataset_like
        tokenizer = getattr(base_dataset, "tokenizer", None)
        return base_dataset, tokenizer
    return None, None

base_dataset, tokenizer = _resolve_runtime_dataset_and_tokenizer(data_cfg)
if tokenizer is None:
    raise ValueError("Could not resolve tokenizer from data_cfg runtime datasets.")

attack_vectors = getattr(attack_cfg, "attack", None)
if attack_vectors is None:
    raise ValueError("attack_cfg.attack is empty. Run the attack cell first.")

if not isinstance(attack_vectors, (list, tuple)):
    attack_vectors = [attack_vectors]

rows = []
for idx, attacked in enumerate(attack_vectors):
    clean_sample = None
    if base_dataset is not None and hasattr(base_dataset, "__getitem__") and idx < len(base_dataset):
        clean_sample = base_dataset[idx]

    rows.append({
        "index": idx,
        "clean_text": _decode_text(tokenizer, clean_sample) if clean_sample is not None else None,
        "attacked_text": _decode_text(tokenizer, attacked),
    })

{
    "num_attack_vectors": len(attack_vectors),
    "side_by_side_preview": rows[:10],
}

In [ ]:
import pandas as pd

def _resolve_runtime_dataset_and_tokenizer_local(data_config):
    if hasattr(data_config, "_X") and isinstance(data_config._X, (tuple, list)) and len(data_config._X) >= 2:
        dataset_like = data_config._X[1]
        base = dataset_like.dataset if hasattr(dataset_like, "dataset") else dataset_like
        tok = getattr(base, "tokenizer", None)
        return base, tok
    return None, None

base_dataset_df, tokenizer_df = _resolve_runtime_dataset_and_tokenizer_local(data_cfg)
if tokenizer_df is None:
    raise ValueError("Could not resolve tokenizer from data_cfg runtime datasets.")

def _rows_for_attack(attack_obj, attack_label, limit=10):
    attack_vectors_local = getattr(attack_obj, "attack", None)
    if attack_vectors_local is None:
        return []
    if not isinstance(attack_vectors_local, (list, tuple)):
        attack_vectors_local = [attack_vectors_local]

    local_rows = []
    for idx, attacked in enumerate(attack_vectors_local[:limit]):
        clean_sample = None
        if base_dataset_df is not None and hasattr(base_dataset_df, "__getitem__") and idx < len(base_dataset_df):
            clean_sample = base_dataset_df[idx]
        local_rows.append(
            {
                "attack_family": attack_label,
                "index": idx,
                "clean_text": _decode_text(tokenizer_df, clean_sample) if clean_sample is not None else None,
                "attacked_text": _decode_text(tokenizer_df, attacked),
            }
        )
    return local_rows

rows_textattack = _rows_for_attack(textattack_experiment.attack, "textattack", limit=10)
rows_openattack = _rows_for_attack(openattack_experiment.attack, "openattack", limit=10)

comparison_df = pd.DataFrame(rows_textattack + rows_openattack)
comparison_df["changed"] = comparison_df["clean_text"] != comparison_df["attacked_text"]
comparison_df